<a href="https://colab.research.google.com/github/tbkashin/Machine-Learning/blob/Homework/Homework_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question 1

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

# Load and preprocess the CIFAR-10 dataset
# The problem specifies using the test set directly as the validation data
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0


# Part (i): Construct the CNN Architecture

model = models.Sequential()

# a) 2D convolutional layer with 64 filters, size (5, 5), ReLU, and input shape for CIFAR-10 (32x32x3)
model.add(layers.Conv2D(64, (5, 5), activation='relu', input_shape=(32, 32, 3)))

# b) Max pooling layer of size (2, 2)
model.add(layers.MaxPooling2D((2, 2)))

# c) 2D convolutional layer with 32 filters, size (3, 3), ReLU
model.add(layers.Conv2D(32, (3, 3), activation='relu'))

# d) Max pooling layer of size (2, 2)
model.add(layers.MaxPooling2D((2, 2)))

# e) 2D convolutional layer with 32 filters, size (3, 3), ReLU
model.add(layers.Conv2D(32, (3, 3), activation='relu'))

# f) Flatten layer to convert to a one-dimensional vector
model.add(layers.Flatten())

# g) Dense layer with 64 neurons and ReLU activation
model.add(layers.Dense(64, activation='relu'))

# h) Final layer producing estimated probabilities for the 10 CIFAR-10 classes
model.add(layers.Dense(10, activation='softmax'))

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Display the model architecture
model.summary()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1618s 9us/step


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 64)     │         4,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 12, 12, 32)     │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 4, 4, 32)       │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 66,058 (258.04 KB)

 Trainable params: 66,058 (258.04 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Part (ii): Train and Plot Accuracies

# Train for 10 epochs with a batch size of 64
history = model.fit(train_images, train_labels,
                    epochs=10,
                    batch_size=64,
                    validation_data=(test_images, test_labels))

# Calculate and plot the training and validation accuracies
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o')
plt.title('CNN Training and Validation Accuracy over 10 Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# Question 2

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Load the Scikit-learn breast cancer Wisconsin dataset
cancer_data = load_breast_cancer()
df = pd.DataFrame(cancer_data.data, columns=cancer_data.feature_names)
y = cancer_data.target

# Isolate the exact features requested for classification
selected_features = ['worst compactness', 'worst concavity', 'worst area']
X = df[selected_features]

# Split the dataset into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standardize the features (SVMs are highly sensitive to feature scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Part (i): SVM with Linear Kernel

svm_linear = SVC(kernel='linear', random_state=42)
svm_linear.fit(X_train_scaled, y_train)

y_pred_linear = svm_linear.predict(X_test_scaled)
acc_linear = accuracy_score(y_test, y_pred_linear)
print(f"Part (i) - Accuracy with Linear Kernel: {acc_linear:.4f}")


# Part (ii): SVM with RBF Kernel (C = 2^7)

# C = 2^7 corresponds to C = 128
svm_rbf = SVC(kernel='rbf', C=2**7, random_state=42)
svm_rbf.fit(X_train_scaled, y_train)

y_pred_rbf = svm_rbf.predict(X_test_scaled)
acc_rbf = accuracy_score(y_test, y_pred_rbf)
print(f"Part (ii) - Accuracy with RBF Kernel (C=128): {acc_rbf:.4f}")